In [2]:
#import libraries
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
import tensorflow as tf
from tensorflow.keras import layers
import pandas as pd

In [3]:
# Charger le fichier (IMPORTANT : separator = tab)
df = pd.read_csv("amazon_cells_labelled_LARGE_25K.txt", sep="\t", header=None)

# Renommer colonnes
df.columns = ["text", "label"]

# Séparer
X = df["text"]
y = df["label"]

# Train + temp (val + test)
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)

# Validation + test
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

print(len(X_train), len(X_val), len(X_test))


print(X.head())
print(y.head())

17500 3750 3750
0    I've read this book with much expectation, it ...
1    love it...very touch.it's to bad that in the d...
2    The creepiest book I've ever read! It's a cree...
3    It starts off a bit slow, but once the product...
4    As good as this book may be, the print quality...
Name: text, dtype: object
0    0
1    1
2    1
3    1
4    0
Name: label, dtype: int64


In [4]:
vectorizer = TfidfVectorizer(max_features=5000)

X_train_vec = vectorizer.fit_transform(X_train).toarray()
X_val_vec   = vectorizer.transform(X_val).toarray()
X_test_vec  = vectorizer.transform(X_test).toarray()

In [5]:
model_ann = tf.keras.Sequential([
    layers.Dense(128, activation='relu', input_shape=(5000,)),
    layers.Dropout(0.5),
    layers.Dense(64, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

2026-04-08 13:27:06.525874: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M3
2026-04-08 13:27:06.525911: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 24.00 GB
2026-04-08 13:27:06.525918: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 8.88 GB
2026-04-08 13:27:06.526021: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:306] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-04-08 13:27:06.526153: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:272] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


In [6]:
model_ann.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [7]:
model_ann.fit(
    X_train_vec, y_train,
    validation_data=(X_val_vec, y_val),
    epochs=10,
    batch_size=32
)

Epoch 1/10


: 